In [ ]:
import pandas as pd
import numpy as np
from nltk.corpus import words as nltk_words
import sys
# Add the parent directory to Python's module search path
sys.path.append('..')
from src.run import run_steps
from src.steps import load_words_to_set
import concurrent.futures
from functools import partial
import os
from transformers import (
    pipeline,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TextGenerationPipeline,
)
import torch
from steps import (
    extract_sentiment_label,
    score_from_label,
    generate_text
)

In [5]:
df = pd.read_excel('C:/Users/HP/Desktop/tiktok-analysis/dataset_tiktok-comments-scraper_2026-04-22_19-27-52-195.xlsx')

In [6]:
df["commentText"] = df["commentText"].replace(r'^\s*$', np.nan, regex=True)
df = df.dropna(subset=["commentText"])

In [7]:
comments_str = df["commentText"].str.cat(sep=' ')

In [8]:
words_dict_set: dict[str, set[str]] = {
        "English": set(nltk_words.words()),
        "Luganda": load_words_to_set(
            file_path="C:/Users/HP/Desktop/tiktok-analysis/words/processed/luganda_words.txt"
        ),
        "Swahili": load_words_to_set(
            file_path="C:/Users/HP/Desktop/tiktok-analysis/words/processed/swahili_words.txt"
        ),
    }

In [9]:
words_dict_list: dict[str, list[str]] = {
        "English": list(words_dict_set["English"]),
        "Luganda": list(words_dict_set["Luganda"]),
        "Swahili": list(words_dict_set["Swahili"])
    }

In [10]:
run_steps(initial_word="nyabo", words_dict_set=words_dict_set, words_dict_list = words_dict_list)

('Luganda', ('nnyabo', 90.9090909090909, 166))

I honestly think fuzzy match can handle at least 90% of misspellings or unknown words from the lexicon lookups... So we may not need the n-grams classifier after all... but let's run it through all the comment words and then we get the percentage that after passing "run_steps" on, returns "None"... Only if a small percentage is caught will I consider the classifier...

In [11]:
all_words = comments_str.split()
unique_words = list(set(all_words))

# Prepare the target function
# The executor's map() function expects a target function that takes exactly ONE argument (the word).
# We use partial() to "freeze" your dictionary arguments into the function ahead of time... so we will be inputting just one.
run_steps_partial = partial(
    run_steps,
    words_dict_set=words_dict_set,
    words_dict_list=words_dict_list
)

max_cores = os.cpu_count() or 12
print(f"Processing {len(unique_words)} across {max_cores} cores...")

# 3. Process in parallel
# ProcessPoolExecutor creates separate Python processes to bypass Python's single-core limitation
with concurrent.futures.ProcessPoolExecutor(max_workers=max_cores) as executor:
    # executor.map takes our modified function and applies it to every unique word
    results_list = list(executor.map(run_steps_partial, unique_words))

# Create the O(1) lookup
word_results_lookup = dict(zip(unique_words, results_list))


percent_list = []
count = 0

for word in all_words:
    result = word_results_lookup[word]
    percent_list.append(result)
    if result == "None":
        count = count + 1

Processing 3233 across 12 cores...


In [12]:
none_percent = (count/len(percent_list))*100
print("Percentage of 'None' results: ",none_percent)

Percentage of 'None' results:  10.377430656487615


Definitely not worth the trouble!

Now let's do the proper runthorugh for ALL words. Execute the CPU bound tasks

In [ ]:
all_words = comments_str.split()

# Prepare the target function
# The executor's map() function expects a target function that takes exactly ONE argument (the word).
# We use partial() to "freeze" your dictionary arguments into the function ahead of time... so we will be inputting just one.
run_steps_partial = partial(
    run_steps,
    words_dict_set=words_dict_set,
    words_dict_list=words_dict_list
)

max_cores = os.cpu_count() or 12
print(f"Processing {len(all_words)} across {max_cores} cores...")

# 3. Process in parallel
# ProcessPoolExecutor creates separate Python processes to bypass Python's single-core limitation
with concurrent.futures.ProcessPoolExecutor(max_workers=max_cores) as executor:
    # executor.map takes our modified function and applies it to every word
    results_list = list(executor.map(run_steps_partial, all_words))

# Create the O(1) lookup
#word_results_lookup = dict(zip(all_words, results_list))

Execute the GPU bound tasks

In [ ]:
generation_pipelines: dict = {"English": None, "Luganda": None | TextGenerationPipeline, "Swahili": None | TextGenerationPipeline}

model_g_path = "C:/Users/HP/ganda-gemma-1b"
model_s_path = "C:/Users/HP/swahili-gemma-1b"

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

# config to share across models
load_args = {
    "device_map": "auto",
    "quantization_config": quantization_config,
    "low_cpu_mem_usage": True,
}

In [ ]:
pending_luganda_words = []
pending_swahili_words = []

In [ ]:
for result in results_list:
    if result["status"] == "pending_gpu":
        if result["checked_lang"] == "Luganda":
            pending_luganda_words.append(result["checked_text"])
        else:
            pending_swahili_words.append(result["checked_text"])

In [ ]:
lang_name = result["checked_lang"]
if generation_pipelines[lang_name] is None:
    model_path = model_g_path if lang_name == "Luganda" else model_s_path
    model = AutoModelForCausalLM.from_pretrained(model_path, **load_args)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    gen_pipe = generation_pipelines[lang_name] = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device=0 if torch.cuda.is_available() else -1,
    )



# Luganda/Swahili path: use your local Gemma models for correction and sentiment.

sentiment_prompt = (
    f"Classify the sentiment of this {lang_name} text. "
    f"Answer with one word only: positive, neutral, or negative. "
    f"Text: {result["checked_text"]}"
)

sentiment_prompt = [lang_name, result["checked_text"] for]

sentiment_raw = generate_text(gen_pipe, sentiment_prompt)
sentiment_label = extract_sentiment_label(sentiment_raw)
sentiment_score = score_from_label(sentiment_label)

result["polarity_label"] = sentiment_label
result["polarity_score"] = sentiment_score

Put it all in a df

In [ ]:
# put in a dataframe, put together repeating words and add a column for term frequency
words = pd.DataFrame(data=results_list, columns=["corrected_text", "checked_lang", "polarity_score", "polarity_label"])

words.rename(columns={"corrected_text": "Word", "checked_lang": "Language", "polarity_score": "Polarity Score", "polarity_label": "Polarity Label"}, inplace=True)

# to add term frequency to df
words = (
    words.groupby(["Word", "Language", "Polarity Score", "Polarity Label"]).size().reset_index(name="Term Frequency")
)